## tách file train trộn data generative và gốc và test 100% gốc

In [1]:
import pandas as pd
import numpy as np

# --- CẤU HÌNH ---
target_col = 'Yield'  # <--- HÃY ĐỔI TÊN NÀY THÀNH CỘT BẠN MUỐN DỰ ĐOÁN (VD: Yield, Productivity...)
target_orig_count = 178
target_v6_count = 15822
file_orig = 'Agri_Data_Cleaned.csv'
file_v6 = 'Agri_Data_SMOTE_Noise_v6.csv'

# 1. Load dữ liệu
try:
    df_orig = pd.read_csv(file_orig)
    df_v6 = pd.read_csv(file_v6)
except FileNotFoundError:
    print("Không tìm thấy file csv. Hãy kiểm tra lại đường dẫn.")
    # Dùng dữ liệu giả lập để demo code chạy (nếu bạn copy paste chạy thử)
    # df_orig = pd.DataFrame(...) 

print(f"Tổng số dòng data gốc: {len(df_orig)}")

# 2. Chiến thuật lấy 179 dòng TỐI ƯU HÓA PHÂN PHỐI (Distribution Aware Stratified Sampling)
selected_indices = set()

# --- BƯỚC 1: Bắt buộc lấy các giá trị CAO NHẤT (Top Values) ---
# Mục đích: Đưa Max của Train lên bằng Max của tổng thể (98.99)
# Lấy top 5% hoặc top 20 dòng có giá trị cao nhất để mô hình học được các trường hợp cực đoan
top_n = 20 
top_indices = df_orig.nlargest(top_n, target_col).index
selected_indices.update(top_indices)

# --- BƯỚC 2: Lấy mỗi District ít nhất 1 mẫu (ưu tiên mẫu chưa chọn) ---
groups_district = df_orig.groupby('District')
for name, group in groups_district:
    # Chỉ lấy nếu nhóm này chưa có đại diện nào trong danh sách đã chọn
    if not any(idx in selected_indices for idx in group.index):
        # Ưu tiên lấy mẫu có giá trị target cao/trung bình thay vì thấp để kéo Mean lên
        # Hoặc lấy ngẫu nhiên
        idx = np.random.choice(group.index, 1)[0]
        selected_indices.add(idx)

# --- BƯỚC 3: Lấy mỗi Crop Name ít nhất 1 mẫu ---
current_crops = df_orig.loc[list(selected_indices), 'Crop Name'].unique()
missing_crops = set(df_orig['Crop Name'].unique()) - set(current_crops)

for crop in missing_crops:
    candidates = df_orig[df_orig['Crop Name'] == crop].index
    # Tránh lấy trùng
    new_candidates = list(set(candidates) - selected_indices)
    if new_candidates:
        # Nếu được, hãy chọn candidate có giá trị target cao để tăng độ khó cho train
        idx = np.random.choice(new_candidates, 1)[0]
        selected_indices.add(idx)

# --- BƯỚC 4: Điền cho đủ 179 dòng (Stratified Binning nếu có thể, hoặc Random) ---
current_count = len(selected_indices)
if current_count < target_orig_count:
    needed = target_orig_count - current_count
    remaining_pool = list(set(df_orig.index) - selected_indices)
    
    # Để đảm bảo Mean và Std của Train gần với Test hơn, ta nên random
    # trong tập còn lại. Do Bước 1 đã lấy Top Max, nên phần còn lại lấy random là ổn.
    extra_indices = np.random.choice(remaining_pool, needed, replace=False)
    selected_indices.update(extra_indices)

# Chuyển thành list và tạo DataFrame
final_orig_indices = list(selected_indices)
df_train_orig_part = df_orig.loc[final_orig_indices]
df_test_final = df_orig.drop(index=final_orig_indices)

# --- KIỂM TRA CHỈ SỐ THỐNG KÊ (QUAN TRỌNG) ---
print("-" * 30)
print("THỐNG KÊ SAU KHI CHIA (DATA GỐC):")
print(f"Train (179 dòng) - Mean: {df_train_orig_part[target_col].mean():.2f}, "
      f"Std: {df_train_orig_part[target_col].std():.2f}, "
      f"Max: {df_train_orig_part[target_col].max():.2f}")
print(f"Test (Còn lại)   - Mean: {df_test_final[target_col].mean():.2f}, "
      f"Std: {df_test_final[target_col].std():.2f}, "
      f"Max: {df_test_final[target_col].max():.2f}")
print("-" * 30)

# 3. Xử lý file v6 (Lấy 15821 dòng)
if len(df_v6) >= target_v6_count:
    df_train_v6_part = df_v6.sample(n=target_v6_count, random_state=42)
else:
    print(f"Cảnh báo: v6 chỉ có {len(df_v6)} dòng, lấy toàn bộ.")
    df_train_v6_part = df_v6.copy()

# 4. Ghép lại thành tập Train hoàn chỉnh
df_train_final = pd.concat([df_train_orig_part, df_train_v6_part], ignore_index=True)
df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Kích thước tập Train cuối cùng: {df_train_final.shape}")

# 5. Lưu file
df_train_final.to_csv('Agri_Train_Combined_16k.csv', index=False)
df_test_final.to_csv('Agri_Test_Original_4k.csv', index=False)

print("Đã xuất file: Agri_Train_Combined_16k_Optimized.csv và Agri_Test_Original_4k_Optimized.csv")

Tổng số dòng data gốc: 4178
------------------------------
THỐNG KÊ SAU KHI CHIA (DATA GỐC):
Train (179 dòng) - Mean: 9.00, Std: 16.57, Max: 98.99
Test (Còn lại)   - Mean: 3.96, Std: 4.65, Max: 34.66
------------------------------
Kích thước tập Train cuối cùng: (16000, 51)
Đã xuất file: Agri_Train_Combined_16k_Optimized.csv và Agri_Test_Original_4k_Optimized.csv
